## 1. Read data

### 1.1. The dataset of sentences

In [1]:
import pandas as pd
sentence_df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_datasets/translated_sentence_data.xlsx")

In [2]:
sentences = sentence_df["Clean_Sentence"].to_list()

### 1.2. Import list of stopwords

In [3]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/mijnidbcoachnlp/data/analysis_datasets/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec', "jl", "weken", "week", "dagen", "dag", "mg"
]

dutch_stopwords.extend(extra_list)

## 2. Download the embedding model BERTje

### 2.1 Download BERTje

In [4]:
from transformers.pipelines import pipeline
from transformers import AutoTokenizer, AutoModel, TFAutoModel
embedding_model = pipeline("feature-extraction", model="GroNLP/bert-base-dutch-cased")

Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 2.2 Pre-calculate and save sentence embeddings (skip if there are saved embeddings and go to 2.3)

In [ ]:
# pre-calculate the embeddings of the dutch sentences
import numpy as np

embeddings_nl = embedding_model(sentences, truncation=True, padding=True)
sentence_embeddings_nl = np.array([np.mean(embedding, axis=1).flatten() for embedding in embeddings_nl])

In [ ]:
from tqdm import tqdm 
# Initialize an empty list to store embeddings
sentence_embeddings_nl = []

# Use tqdm to track progress
for sentence in tqdm(sentences, desc="Processing Embeddings", unit="sentence"):
    # Compute embedding for the sentence
    embedding = embedding_model(sentence)
    # Average token embeddings to create a sentence-level embedding
    sentence_embedding = np.mean(embedding[0], axis=0)
    # Append to the list
    sentence_embeddings_nl.append(sentence_embedding)

# Convert the list to a NumPy array
sentence_embeddings_nl = np.array(sentence_embeddings_nl)

# Check the shape of the resulting embeddings
print(f"Shape of sentence embeddings: {sentence_embeddings_nl.shape}")

In [ ]:
# save the sentence embeddings
import numpy as np
np.save('/workspace/mijnidbcoachnlp/data/embeddings/sentence_embeddings_bertje_nl_new.npy', sentence_embeddings_nl)

### 2.3 Load saved embeddings

In [30]:
# load the save model
import numpy as np

loaded_embeddings_nl = np.load('/workspace/mijnidbcoachnlp/data/embeddings/sentence_embeddings_bertje_nl_new.npy')

In [31]:
embeddings=loaded_embeddings_nl

## 3. Preparation for Model Evaluation

### 3.1 Import the annotated sample set

In [7]:
annotated_df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_datasets/sample_sentences_for_labeling_100.xlsx", index_col=0)

### 3.2 Function to build model with different clustering methods

In [16]:
# set the vectorizer model
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired


kmeans_model = KMeans(n_clusters=100)
agglo_model = AgglomerativeClustering(n_clusters=100)
# dimension reduction
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = KeyBERTInspired()


In [32]:
# function to build the model with different selection of clustering method
# build the BERTopic model
from bertopic import BERTopic
from umap import UMAP
from bertopic import BERTopic

def build_bertopic():
    # Initialize BERTopic with UMAP and HDBSCAN models
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=UMAP(n_neighbors=10, n_components=2, metric='cosine', random_state=42),
        hdbscan_model=HDBSCAN(min_cluster_size=20, min_samples=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True),
        vectorizer_model=CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 2), token_pattern=r'\b[a-zA-Z]{3,}\b'),
        top_n_words=10,
        verbose=True
    )

    # Fit-transform to get document topics and probabilities
    topics, probs = topic_model.fit_transform(sentences, embeddings)
    

    # Return only essential results
    return topic_model

## 4. BERTje-HDBSCAN model

In [33]:
topic_model = build_bertopic()

2024-11-18 10:35:17,405 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-11-18 10:36:05,340 - BERTopic - Dimensionality - Completed ✓
2024-11-18 10:36:05,343 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-11-18 10:36:07,263 - BERTopic - Cluster - Completed ✓
2024-11-18 10:36:07,279 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-11-18 10:36:08,549 - BERTopic - Representation - Completed ✓


In [35]:
# save topic model 
topic_model.save("/workspace/mijnidbcoachnlp/data/result_data/saved_BERTje_model", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)


In [39]:
loaded_topic_model = BERTopic.load("/workspace/mijnidbcoachnlp/data/result_data/saved_BERTje_model", embedding_model=embedding_model)

In [36]:
topic_model.get_topic_info()[:10]

,Topic,Count,Name,Representation,Representative_Docs
0,-1,18991,-1_afspraak_goed_medicatie_ontlasting,"[afspraak, goed, medicatie, ontlasting, vraag,...",[Afgelopen donderdag heb ik de tijdens de remi...
1,0,1820,0_toilet_buikpijn_pijn_buik,"[toilet, buikpijn, pijn, buik, gevoel, last, k...",[Wel moet ik de afgelopen weken vaker naar het...
2,1,694,1_sturen_wilt_recept apotheek_recept sturen,"[sturen, wilt, recept apotheek, recept sturen,...",[Wilt u een recept naar de apotheek in sturen...
3,2,478,2_nieuw recept_nieuw_recept_graag nieuw,"[nieuw recept, nieuw, recept, graag nieuw, her...",[Ik wil graag weer een nieuw recept puri-netho...
4,3,443,3_ziekenhuis_apotheek ziekenhuis_ziekenhuis ap...,"[ziekenhuis, apotheek ziekenhuis, ziekenhuis a...",[8 januari moet ik weer naar het [ZIEKENHUIS]....
5,4,350,4_snelle_snelle reactie_bedankt_reactie,"[snelle, snelle reactie, bedankt, reactie, bed...","[Heel erg bedankt voor je snelle reactie!, Hsa..."
6,5,334,5_infusie_vraag_graag verneem_overleg,"[infusie, vraag, graag verneem, overleg, nadel...","[Daarom overleg ik graag vandaag nog (d.w.z., ..."
7,6,310,6_last_laatste last_last darmen_buikpijn,"[last, laatste last, last darmen, buikpijn, bu...","[Heb al een twee weken weer last., Terwijl ik ..."
8,7,294,7_ontlasting ingeleverd_ingeleverd_ingeleverd ...,"[ontlasting ingeleverd, ingeleverd, ingeleverd...",[Op heb ik bloed laten prikken en ontlasting ...
9,8,277,8_apotheek_apotheek bestellen_kijken_chronisch,"[apotheek, apotheek bestellen, kijken, chronis...",[Om dit te voorkomen heb ja vaker een antibiot...


In [40]:
# Reduce outliers
topics = topic_model.topics_
new_topics = topic_model.reduce_outliers(sentences, topics)

100%|██████████| 19/19 [00:02<00:00,  8.06it/s]


In [41]:
topic_model.update_topics(sentences, topics=new_topics)

2024-11-18 10:38:04,163 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [44]:
topic_model.save("/workspace/mijnidbcoachnlp/data/result_data/saved_BERTje_model_reduced_outliers", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

In [143]:
loaded_model_ro = BERTopic.load("/workspace/mijnidbcoachnlp/data/result_data/saved_BERTje_model_reduced_outliers", embedding_model=embedding_model)

In [147]:
# INSPECT topics again # Looks good
topic_model = loaded_model_ro
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,240,-1_jl_topper_bent_ja,"[jl, topper, bent, ja, braakgevoel, vriendekijke, beangstigend, 09, doen, 1x]",NaN
1,0,2231,0_pijn_toilet_buikpijn_gevoel,"[pijn, toilet, buikpijn, gevoel, buik, last, veel, wc, diarree, krampen]",NaN
2,1,828,1_kunt_zouden_kunnen_sturen,"[kunt, zouden, kunnen, sturen, mij, doorgeven, jullie, willen, wilt, recept]",NaN
3,2,834,2_recept_nieuw_herhaalrecept_nethol,"[recept, nieuw, herhaalrecept, nethol, aanvragen, herhaal, puri, apotheek, pentasa, graag]",NaN
4,3,603,3_ziekenhuis_in_apotheek_naar,"[ziekenhuis, in, apotheek, naar, het, bij, gestuurd, komen, prikken, ophalen]",NaN
...,...,...,...,...,...
335,334,46,334_nachts_extreem_wakker_zondagochtend,"[nachts, extreem, wakker, zondagochtend, maagzuur, gespuugd, warm, hierdoor, borrelt, gewrichten]",NaN
336,335,46,335_voet_irritatie_gewricht_aantal,"[voet, irritatie, gewricht, aantal, sinds, heftige, donderdagavond, warmte, enkels, bepaalde]",NaN
337,336,21,336_uw_alvast_reactie_bedankt,"[uw, alvast, reactie, bedankt, antwoord, voor, advies, jullie, en, ]",NaN
338,337,49,337_calpro_test_gedaan_gluten,"[calpro, test, gedaan, gluten, calprotectine, abces, achtergelaten, ug, score, conform]",NaN


In [46]:
#save the topic info
topic_info = topic_model.get_topic_info()
topic_info.to_csv('/workspace/mijnidbcoachnlp/data/result_data/topic_info_bertje.csv', index=False)

In [47]:
#save the topic info
doc_info = topic_model.get_document_info(sentences)
doc_info.to_csv('/workspace/mijnidbcoachnlp/data/result_data/doc_info_bertje.csv', index=False)

### 4.2 Merging topics with hierarchy

In [169]:
from sklearn.feature_extraction.text import CountVectorizer

# configure the sub-models
#vectorizer
vectorizer_model=CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 2), token_pattern=r'\b[a-zA-Z]{3,}\b')
topic_model.update_topics(sentences, vectorizer_model=vectorizer_model)


In [170]:
topic_model.save("/workspace/mijnidbcoachnlp/data/result_data/saved_BERTje_model_updated_topics", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)


In [201]:
topic_model = BERTopic.load("/workspace/mijnidbcoachnlp/data/result_data/saved_BERTje_model_updated_topics", embedding_model=embedding_model)

In [258]:
hierarchical_topics = topic_model.hierarchical_topics(sentences)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 89/89 [00:00<00:00, 164.87it/s]


In [259]:
#save the topic info
topic_info = topic_model.get_topic_info()
topic_info["Name"].to_list()

['-1_bedankt_dank_reactie_hoor',
 '0_recept_apotheek_ziekenhuis_nieuw',
 '1_last_pijn_buikpijn_ontlasting',
 '2_prikken_bloed_laten prikken_bloed laten',
 '3_afspraak_contact_telefonisch_gebeld',
 '4_gebruik_infuus_paracetamol_daags',
 '5_formulieren_formulier_potje_ontvangen',
 '6_gaat_voel_beter_goed',
 '7_advies_graag_graag advies_graag reactie',
 '8_endoscopie_coloscopie_scan_mri',
 '9_huisarts_dermatoloog_spreekuur_klinische',
 '10_nummer_vaccin_combinatie_coronavirus',
 '11_crohn_ziekte_ziekte crohn_medicatie',
 '12_stress_kilo_afgevallen_eten',
 '13_zwanger_begonnen_zwangerschapstest_interval',
 '14_klysmas_denk_laag_voel',
 '15_onderzoek_darmkanker_akkoord_bevolkingsonderzoek',
 '16_toegenomen_vorig jaar_vorig_humira',
 '17_vind_lang_duren_vervelend',
 '18_heden_gehoord_vernomen_heden ontvangen',
 '19_loop_ene_begint_toegenomen',
 '20_verstandig_handig_raadzaam_wachten',
 '21_wacht_telefoontje_afwachten_wacht telefoontje',
 '22_optie_nodig_alternatief_gevolgen',
 '23_zorgen_ong

In [257]:
# merge some topics
topics_to_merge = [[-1, 17, 18, 26, 29, 34, 35, 36, 40, 42, 43, 44, 48], #other
                    [0], # pharmacy
                    [1, 14, 19, 28, 30, 50], # symptom
                    [2, 32, 39, 49], # test
                    [3, 21, 25], # appointment
                    [4, 11, 20, 45, 47, 55], # medication
                    [5, 46, ], # administrative communication
                    [6, 16, 27, 33, 41], # general health update
                    [7, 22, 23, 24, 31, 38, 51, 52, 54], #advice/discussion
                    [8, 15], # exam
                    [9, 53], # consultation/hospital visit
                    [10, 37], #vaccination
                    [12], # food
                    [13], # pregnancy
                    [], # work
                    []] # surgery
 
topic_model.merge_topics(sentences, topics_to_merge)  

In [285]:
pd.set_option("max_colwidth", 600)
topic_model.get_topic_info(56)

,Topic,Count,Name,Representation,Representative_Docs
0,56,94,56_mogelijkheid_toevallig_wachten_krijgen wachten,"[mogelijkheid, toevallig, wachten, krijgen wachten, vaccine, afgifte, levering, nieuwe levering, bloedafname, krijgen]","[Is er toevallig een mogelijkheid om voor menssen met croon via een andere weg eerder de vaccine krijgen want moet wachten op de huisarts ., Is er toevallig een mogelijkheid om voor menssen met croon via een andere weg eerder de vaccine krijgen want moet wachten op de huisarts ., Is er toevallig een mogelijkheid om voor menssen met croon via een andere weg eerder de vaccine krijgen want moet wachten op de huisarts .]"


In [277]:
doc_info_new = topic_model.get_document_info(sentences)
doc_info_new[doc_info_new["Topic"] == 31]

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Representative_document
186,En ik moet ook voor een nieuw gebit dat was niet goed meer er worden 5 tanden getrokken in,31,31_sprake_waarvoor_zalf_zalf gekregen,"[sprake, waarvoor, zalf, zalf gekregen, gekregen indien, verbeterd, sprake schimmel, terug komen, schimmel, cardiologie]","[Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd over 6 weken terug komen., Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd over 6 weken terug komen., Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd over 6 weken terug komen.]",sprake - waarvoor - zalf - zalf gekregen - gekregen indien - verbeterd - sprake schimmel - terug komen - schimmel - cardiologie,False
197,dinsdag Hoi Ik heb een vraag hoelang duurt een afspraak bij een cardiologie ik heb het aan mijn hart Maar ik hoor maar niks of moet ik nog wachten ik ben zenuwachtig ze hebben mij onderzoek gedaan Bij polikliniek anesthesiologie voor een totaal nieuwe knie operatie ik ben daar geweest op 7 juli ’15 nu gaan al bloed onderzoeken doen daar moet ik lab boven zijn op afd.,31,31_sprake_waarvoor_zalf_zalf gekregen,"[sprake, waarvoor, zalf, zalf gekregen, gekregen indien, verbeterd, sprake schimmel, terug komen, schimmel, cardiologie]","[Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd over 6 weken terug komen., Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd over 6 weken terug komen., Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd over 6 weken terug komen.]",sprake - waarvoor - zalf - zalf gekregen - gekregen indien - verbeterd - sprake schimmel - terug komen - schimmel - cardiologie,False
204,ik heb toch borstkanker nu zijn ze met een Paclitaxel en Herceptin bezig nu hebben weer een brief vragenlijst gestuurd nu heb ik even geen zin in wat moet ik doen,31,31_sprake_waarvoor_zalf_zalf gekregen,"[sprake, waarvoor, zalf, zalf gekregen, gekregen indien, verbeterd, sprake schimmel, terug komen, schimmel, cardiologie]","[Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd over 6 weken terug komen., Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd over 6 weken terug komen., Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd over 6 weken terug komen.]",sprake - waarvoor - zalf - zalf gekregen - gekregen indien - verbeterd - sprake schimmel - terug komen - schimmel - cardiologie,False
1051,Heb wederom contact met mijn huisarts gehad en volgens deze zou het wel eens aan de mezalasine kunnen liggen.,31,31_sprake_waarvoor_zalf_zalf gekregen,"[sprake, waarvoor, zalf, zalf gekregen, gekregen indien, verbeterd, sprake schimmel, terug komen, schimmel, cardiologie]","[Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd over 6 weken terug komen., Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd over 6 weken terug komen., Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd over 6 weken terug komen.]",sprake - waarvoor - zalf - zalf gekregen - gekregen indien - verbeterd - sprake schimmel - terug komen - schimmel - cardiologie,False
1391,Tijdens het polibezoek van 26 februari gaf ik aan dat het herstel nog niet echt vorderde en dat er nog veel bloed aanwezig was.,31,31_sprake_waarvoor_zalf_zalf gekregen,"[sprake, waarvoor, zalf, zalf gekregen, gekregen indien, verbeterd, sprake schimmel, terug komen, schimmel, cardiologie]","[Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd over 6 weken terug komen., Er was sprake van schimmel en exzeem waarvoor ik zalf heb gekregen.Indien het niet verbeterd o